In [18]:
%%capture
!pip install transformers datasets tabulate

In [ ]:
# Install black
!pip install black[jupyter]

# Format the notebook
!black your_notebook.ipynb

In [19]:
import numpy as np
import os
import torch
import torch.nn.functional as F
import torch.nn as nn
import math
from torch.utils.data import DataLoader
from tabulate import tabulate
from datasets import load_dataset

from tqdm.notebook import tqdm
import torch.optim as optim
from transformers import BertTokenizer
from torch import device
from torch.utils.data import Dataset

This is a template of the notebook that you should complete and enrich with your own code.

First cells will be the same than the ones of the lab on text convolution.

# Data loading


In [20]:
dataset = load_dataset("stanfordnlp/imdb", split="train")
print(dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})


# Pre-processing / Tokenization

This is a very important step. It maybe boring but very important. In this session we will be lazy, but in real life, the time spent on inspecting and cleaning data is never wasted. It is true for text, but also for everything.



In PyTorch, everything is tensor. Words are replaced by indices. A sentence, is therefore a sequence of indices (long integers). In the first HW, you constructed a `WhiteSpaceTokenizer`. Here we will use an already built tokenizer. It is more appropriate to transformers. It relies on sub-word units, and converts everything in lower case. This is not always the best choice, but here it will be sufficient. To quote the documentation, this tokenizer allows you to:
- Tokenize (splitting strings in sub-word token strings), converttokens strings to ids and back, and encoding/decoding (i.e., tokenizing and converting to integers).
- Add new tokens to the vocabulary in a way that is independent of the underlying structure (BPE, SentencePiece…).
- Manage special tokens (like mask, beginning-of-sentence, etc.): adding them, assigning them to attributes in the tokenizer for easy access and making sure they are not split during tokenization.

Here we are going to use the tokenizer from the well known Bert model, that we can directly download.

In [21]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)

In [22]:
def preprocessing_fn(x, tokenizer):
    x["input_ids"] = tokenizer(
        x["text"],
        add_special_tokens=False,
        truncation=True,
        max_length=256,
        padding=False,
        return_attention_mask=False,
    )["input_ids"]
    return x

🚧 **TODO** 🚧

Read the documentation about HuggingFace dataset and complete the code below.
You should:
- Shuffle the dataset
- For computational reasons, use only a total of **10000 samples**.
- Tokenize the dataset with the `preprocessing_fn`. (*Hint: use the `Dataset.map` method from HuggingFace*).
- Keep only the column `input_ids`.
- Make a train/validation split, (**80% / 20%**). Call these dataset `train_set` and `valid_set`.


In [23]:
# Select n_samples samples
n_samples = 10000

# Shuffle the dataset
dataset = dataset.shuffle(seed=42)

# Select n_sampless samples
dataset = dataset.select(range(n_samples))

# Tokenize the dataset
dataset = dataset.map(
    lambda x: preprocessing_fn(x, tokenizer),
)

# Remove useless columns
dataset = dataset.remove_columns(["text", "label"])

# Split into train (80%) and validation (20%)
split = dataset.train_test_split(test_size=0.2, seed=42)

document_train_set = split["train"]
document_valid_set = split["test"]

# Verify
print(document_train_set)
print(document_valid_set)
print("Sample input_ids:", document_train_set[0]["input_ids"][:10])

Dataset({
    features: ['input_ids'],
    num_rows: 8000
})
Dataset({
    features: ['input_ids'],
    num_rows: 2000
})
Sample input_ids: [1045, 4669, 27368, 1999, 2395, 4959, 1010, 1998, 1045, 6618]


In [24]:
def extract_words_contexts(doc_ids, R):
    """
    Extract all valid (word, context) pairs from a single document.
    
    Args:
        doc_ids: list of token ids representing a document
        R: radius of the context window
    
    Returns:
        words: list of word ids w
        contexts: list of context ids C+ (each of size 2R)
    
    Border strategy: we ignore words that are too close to the edges,
    i.e. words at index i < R or i >= len(doc) - R. This ensures every
    C+ has exactly 2R elements without introducing artificial padding tokens.
    """
    words = []
    contexts = []

    for i in range(R, len(doc_ids) - R):
        # Word w
        w = doc_ids[i]

        # Positive context C+ : left window + right window (skip w itself)
        left  = doc_ids[i - R : i]        # R words to the left
        right = doc_ids[i + 1 : i + R + 1] # R words to the right
        context = left + right             # 2R words total

        words.append(w)
        contexts.append(context)

    return words, contexts

In [25]:
# Sanity check on one document
sample_doc = document_train_set[0]["input_ids"]
R = 2

words, contexts = extract_words_contexts(sample_doc, R)

print(f"Document length     : {len(sample_doc)}")
print(f"Number of pairs     : {len(words)}")  # should be len(doc) - 2R
print(f"Context size (2R)   : {len(contexts[0])}")  # should be 2R = 4
print(f"Sample word id      : {words[0]}")
print(f"Sample context ids  : {contexts[0]}")

Document length     : 173
Number of pairs     : 169
Context size (2R)   : 4
Sample word id      : 27368
Sample context ids  : [1045, 4669, 1999, 2395]


In [26]:
def flatten_dataset_to_list(dataset, R):
    """
    Apply extract_words_contexts on every document in the dataset
    and flatten the results into two global lists.

    Args:
        dataset: HuggingFace dataset with column "input_ids"
        R: radius of the context window

    Returns:
        all_words   : flat list of all word ids
        all_contexts: flat list of all context ids (each of size 2R)
    """
    all_words = []
    all_contexts = []

    for sample in tqdm(dataset, desc="Flattening dataset"):
        doc_ids = sample["input_ids"]
        words, contexts = extract_words_contexts(doc_ids, R)
        all_words.extend(words)
        all_contexts.extend(contexts)

    return all_words, all_contexts

In [27]:
R = 2

# Apply on train and validation sets
train_words, train_contexts = flatten_dataset_to_list(document_train_set, R)
valid_words, valid_contexts = flatten_dataset_to_list(document_valid_set, R)

# Sanity check
print(f"R                         : {R}")
print(f"Train words               : {len(train_words)}")
print(f"Train contexts            : {len(train_contexts)}")
print(f"Context size (2R)         : {len(train_contexts[0])}")
print(f"Valid words               : {len(valid_words)}")
print(f"Valid contexts            : {len(valid_contexts)}")

# Sample
print(f"\nSample word id            : {train_words[0]}")
print(f"Sample context ids        : {train_contexts[0]}")

Flattening dataset:   0%|          | 0/8000 [00:00<?, ?it/s]

Flattening dataset:   0%|          | 0/2000 [00:00<?, ?it/s]

R                         : 2
Train words               : 1621630
Train contexts            : 1621630
Context size (2R)         : 4
Valid words               : 402063
Valid contexts            : 402063

Sample word id            : 27368
Sample context ids        : [1045, 4669, 1999, 2395]


In [28]:
class Word2VecDataset(Dataset):
    """
    PyTorch Dataset for Word2Vec training.

    Args:
        words   : flat list of word ids
        contexts: flat list of positive context ids (each of size 2R)
    """

    def __init__(self, words, contexts):
        self.words    = words
        self.contexts = contexts

    def __len__(self):
        return len(self.words)

    def __getitem__(self, idx):
        return {
            "word_id"             : torch.tensor(self.words[idx],    dtype=torch.long),
            "positive_context_ids": torch.tensor(self.contexts[idx], dtype=torch.long),
        }


# Instantiate train and valid datasets
train_set = Word2VecDataset(train_words, train_contexts)
valid_set = Word2VecDataset(valid_words, valid_contexts)

# Sanity check
print(f"Train dataset size : {len(train_set)}")
print(f"Valid dataset size : {len(valid_set)}")

sample = train_set[0]
print(f"\nSample word_id              : {sample['word_id']}")
print(f"Sample positive_context_ids : {sample['positive_context_ids']}")
print(f"Shapes -> word_id: {sample['word_id'].shape}, context: {sample['positive_context_ids'].shape}")

Train dataset size : 1621630
Valid dataset size : 402063

Sample word_id              : 27368
Sample positive_context_ids : tensor([1045, 4669, 1999, 2395])
Shapes -> word_id: torch.Size([]), context: torch.Size([4])


In [29]:
def collate_fn(batch, K, vocab_size):
    """
    Custom collate function that adds negative context to each batch.

    Args:
        batch     : list of dicts with keys "word_id" and "positive_context_ids"
        K         : negative sampling ratio (2KR negatives per 2R positives)
        vocab_size: size of the vocabulary to sample negatives from

    Returns:
        dict with keys:
            "word_id"              : (B,)
            "positive_context_ids" : (B, 2R)
            "negative_context_ids" : (B, 2KR)
    """
    # Stack word ids → (B,)
    word_ids = torch.stack([sample["word_id"] for sample in batch])

    # Stack positive contexts → (B, 2R)
    positive_context_ids = torch.stack([sample["positive_context_ids"] for sample in batch])

    # Infer 2R and compute 2KR
    two_R  = positive_context_ids.shape[1]   # 2R
    two_KR = K * two_R                        # 2KR

    B = len(batch)

    # Sample 2KR random words from vocabulary for each example → (B, 2KR)
    negative_context_ids = torch.randint(
        low=0,
        high=vocab_size,
        size=(B, two_KR),
        dtype=torch.long,
    )

    return {
        "word_id"              : word_ids,
        "positive_context_ids" : positive_context_ids,
        "negative_context_ids" : negative_context_ids,
    }

In [30]:
from torch.utils.data import DataLoader
from functools import partial

# Hyperparameters
K          = 5
B          = 64
vocab_size = tokenizer.vocab_size  # 30522 for bert-base-uncased

# Wrap collate_fn with K and vocab_size fixed
collate     = partial(collate_fn, K=K, vocab_size=vocab_size)

train_loader = DataLoader(train_set, batch_size=B, shuffle=True,  collate_fn=collate)
valid_loader = DataLoader(valid_set, batch_size=B, shuffle=False, collate_fn=collate)

In [31]:
print(f"R           : {R}")
print(f"K           : {K}")
print(f"Vocab size  : {vocab_size}")
print(f"Batch size  : {B}\n")

for i, batch in enumerate(train_loader):
    print(f"--- Batch {i+1} ---")
    print(f"word_id              shape : {batch['word_id'].shape}")               # (B,)
    print(f"positive_context_ids shape : {batch['positive_context_ids'].shape}")  # (B, 2R)
    print(f"negative_context_ids shape : {batch['negative_context_ids'].shape}")  # (B, 2KR)
    print()
    if i == 2:
        break

R           : 2
K           : 5
Vocab size  : 30522
Batch size  : 64

--- Batch 1 ---
word_id              shape : torch.Size([64])
positive_context_ids shape : torch.Size([64, 4])
negative_context_ids shape : torch.Size([64, 20])

--- Batch 2 ---
word_id              shape : torch.Size([64])
positive_context_ids shape : torch.Size([64, 4])
negative_context_ids shape : torch.Size([64, 20])

--- Batch 3 ---
word_id              shape : torch.Size([64])
positive_context_ids shape : torch.Size([64, 4])
negative_context_ids shape : torch.Size([64, 20])



# Model

In [32]:
class Word2Vec(nn.Module):
    """
    Word2Vec model with two embedding tables:
        - embedding_w : target word embeddings (Ew)
        - embedding_c : context word embeddings (Ec)

    Args:
        vocab_size: size of the vocabulary |V|
        d         : embedding dimension
    """

    def __init__(self, vocab_size, d):
        super().__init__()

        # Q3.2.1 — Two distinct embedding tables
        self.embedding_w = nn.Embedding(vocab_size, d)  # Ew : target words
        self.embedding_c = nn.Embedding(vocab_size, d)  # Ec : context words

    def forward(self, word_id, positive_context_ids, negative_context_ids):
        """
        Args:
            word_id              : (B,)
            positive_context_ids : (B, 2R)
            negative_context_ids : (B, 2KR)

        Returns:
            loss : scalar
        """

        # --- Step 4 : embed the target word w ---
        w = self.embedding_w(word_id)               # (B, d)

        # --- Step 5 : embed context words ---
        c_pos = self.embedding_c(positive_context_ids)  # (B, 2R,  d)
        c_neg = self.embedding_c(negative_context_ids)  # (B, 2KR, d)

        # --- Step 6 : similarity scores σ(cᵀw) ---
        # We need dot product between w and each context vector
        # w unsqueeze → (B, d, 1) then bmm → (B, 2R, 1) → squeeze → (B, 2R)
        w_unsqueezed = w.unsqueeze(2)                           # (B, d, 1)

        scores_pos = torch.bmm(c_pos, w_unsqueezed).squeeze(2) # (B, 2R)
        scores_neg = torch.bmm(c_neg, w_unsqueezed).squeeze(2) # (B, 2KR)

        # Apply sigmoid
        scores_pos = torch.sigmoid(scores_pos)  # (B, 2R)
        scores_neg = torch.sigmoid(scores_neg)  # (B, 2KR)

        # --- Step 7 : binary cross-entropy loss ---
        # Positives : -log(σ(cᵀw))
        loss_pos = -torch.log(scores_pos + 1e-8)        # (B, 2R)

        # Negatives : -log(1 - σ(cᵀw))
        loss_neg = -torch.log(1 - scores_neg + 1e-8)    # (B, 2KR)

        # Average over all context words and batch
        loss = loss_pos.mean() + loss_neg.mean()

        return loss


# Instantiate the model
d     = 128
model = Word2Vec(vocab_size=vocab_size, d=d)

print(model)
print(f"\nEw shape : {model.embedding_w.weight.shape}")  # (|V|, d)
print(f"Ec shape : {model.embedding_c.weight.shape}")  # (|V|, d)

Word2Vec(
  (embedding_w): Embedding(30522, 128)
  (embedding_c): Embedding(30522, 128)
)

Ew shape : torch.Size([30522, 128])
Ec shape : torch.Size([30522, 128])


In [33]:
def train(model, train_loader, E, device):
    """
    Training loop for Word2Vec model.

    Args:
        model       : Word2Vec model
        train_loader: DataLoader for training set
        E           : number of epochs
        device      : torch device (cpu or cuda)
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(E):
        model.train()
        total_train_loss = 0

        progress_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{E}]")

        for batch in progress_bar:
            word_id              = batch["word_id"].to(device)
            positive_context_ids = batch["positive_context_ids"].to(device)
            negative_context_ids = batch["negative_context_ids"].to(device)

            optimizer.zero_grad()
            loss = model(word_id, positive_context_ids, negative_context_ids)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

            # Live loss update in the progress bar
            progress_bar.set_postfix(loss=loss.item())

        avg_train_loss = total_train_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{E}] | Train Loss : {avg_train_loss:.4f}")

    return model

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = Word2Vec(vocab_size=vocab_size, d=d)
model = train(model, train_loader, 10, device)

Using device: cuda


Epoch [1/10]:   0%|          | 0/25338 [00:00<?, ?it/s]

Epoch [1/10] | Train Loss : 2.7183


Epoch [2/10]:   0%|          | 0/25338 [00:00<?, ?it/s]

Epoch [2/10] | Train Loss : 0.9304


...

Epoch [8/10] | Train Loss : 0.3740


Epoch [9/10]:   0%|          | 0/25338 [00:00<?, ?it/s]

Epoch [9/10] | Train Loss : 0.3586


Epoch [10/10]:   0%|          | 0/25338 [00:00<?, ?it/s]

Epoch [10/10] | Train Loss : 0.3472


In [35]:
def evaluate(model, valid_loader, device):
    """
    Evaluate the Word2Vec model on the test set.

    Accuracy:
        - score > 0.5 for positive context → correct prediction
        - score < 0.5 for negative context → correct prediction

    Args:
        model       : trained Word2Vec model
        valid_loader: DataLoader for the test set
        device      : torch device

    Returns:
        avg_loss : average loss on the test set
        accuracy : accuracy on the test set
    """

    model.eval()
    total_loss     = 0
    total_correct  = 0
    total_examples = 0

    with torch.no_grad():
        for batch in tqdm(valid_loader, desc="Evaluating"):
            word_id              = batch["word_id"].to(device)
            positive_context_ids = batch["positive_context_ids"].to(device)
            negative_context_ids = batch["negative_context_ids"].to(device)

            # --- Loss ---
            loss = model(word_id, positive_context_ids, negative_context_ids)
            total_loss += loss.item()

            # --- Accuracy ---
            w     = model.embedding_w(word_id)                 # (B, d)
            c_pos = model.embedding_c(positive_context_ids)    # (B, 2R, d)
            c_neg = model.embedding_c(negative_context_ids)    # (B, 2KR, d)

            w_unsqueezed = w.unsqueeze(2)                      # (B, d, 1)

            scores_pos = torch.sigmoid(
                torch.bmm(c_pos, w_unsqueezed).squeeze(2)
            )                                                  # (B, 2R)
            scores_neg = torch.sigmoid(
                torch.bmm(c_neg, w_unsqueezed).squeeze(2)
            )                                                  # (B, 2KR)

            # Correct if score > 0.5 for positives
            correct_pos = (scores_pos > 0.5).sum().item()
            # Correct if score < 0.5 for negatives
            correct_neg = (scores_neg < 0.5).sum().item()

            total_correct  += correct_pos + correct_neg
            total_examples += scores_pos.numel() + scores_neg.numel()

    avg_loss = total_loss / len(valid_loader)
    accuracy = total_correct / total_examples

    return avg_loss, accuracy

In [36]:
# --- Run evaluation after training ---
avg_loss, accuracy = evaluate(model, valid_loader, device)
print(f"Test Loss     : {avg_loss:.4f}")
print(f"Test Accuracy : {accuracy:.4f}")

Evaluating:   0%|          | 0/6283 [00:00<?, ?it/s]

Test Loss     : 0.7308
Test Accuracy : 0.9207


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

def save_model(model, d, R, K, B, E, folder="/content/drive/MyDrive/NLP/HW2"):
    """
    Save the Word2Vec embeddings to Google Drive.
    Args:
        model : trained Word2Vec model
        d     : embedding dimension
        R     : context window radius
        K     : negative sampling ratio
        B     : batch size
        E     : number of epochs
        folder: directory to save the checkpoint (default: Google Drive)
    """
    # Create folder if it doesn't exist
    os.makedirs(folder, exist_ok=True)

    # Format filename
    filename = f"model_dim-{d}_radius-{R}_ratio-{K}-batch-{B}-epoch-{E}.ckpt"
    filepath = os.path.join(folder, filename)

    # Save only the embeddings weights
    torch.save({
        "embedding_w": model.embedding_w.weight.data,
        "embedding_c": model.embedding_c.weight.data,
    }, filepath)

    print(f"Model saved to : {filepath}")
    return filepath

Mounted at /content/drive


In [44]:
filepath = save_model(model, d=d, R=R, K=K, B=B, E=10)

Model saved to : /content/drive/MyDrive/NLP/HW2/ckpt/model_dim-128_radius-2_ratio-5-batch-64-epoch-10.ckpt
